# TrustOCT — Training Notebook 1: Baseline Models
### Models: `resnet50` + `msf_resnet50`

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    print('Repository already cloned.')

# Pull latest fixes
!git pull

# Clear cached modules so latest code is used
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith('src'):
        del sys.modules[mod]

try:
    import albumentations, ptflops
except ImportError:
    !pip install -r requirements.txt
print('Setup complete!')

In [ ]:
import os, sys, time, cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully.')
    except Exception as e:
        print(f'Skipped: {e}')

In [ ]:
from src.datasets.dataset import auto_detect_columns, patient_level_split

csv_path = 'kermany_dataset_mapping.csv'

if not os.path.exists(csv_path):
    print('CSV not found. Scanning dataset directories...')
    root_oct = None
    for candidate_root in ['/content/Kermany/OCT2017 ', '/content/Kermany/OCT2017', '/content/Kermany', '/content/OCT2017']:
        if os.path.exists(candidate_root):
            root_oct = candidate_root
            break

    if root_oct:
        import pandas as pd
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)
        print(f'Created CSV with {len(df_new)} images.')
    else:
        print('ERROR: No dataset directory found! Please download the Kermany dataset first.')

if os.path.exists(csv_path):
    import pandas as pd
    df = auto_detect_columns(pd.read_csv(csv_path))

    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'

    if os.path.exists('/content') and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        print('Fixing image paths to local Colab storage...')
        def force_local_path(path_str):
            p = path_str.replace('\\\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath),
                    ]:
                        if os.path.exists(candidate):
                            return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)

    sample_path = df.iloc[0]['image_path']
    print(f'Sample path: {sample_path}')
    print(f'Exists: {os.path.exists(sample_path)}')

    train_df, val_df, test_df = patient_level_split(df)
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
else:
    print('ERROR: Dataset CSV not found and could not be generated!')

In [ ]:
epochs = 30
lr = 1e-4
batch_size = 32

### 1. Train `resnet50` (Baseline)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### 2. Train `msf_resnet50` (+ MultiScale)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### Save Results to GitHub

In [ ]:
import os

# --- PASTE YOUR GITHUB TOKEN BELOW ---
GITHUB_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'  # ← Replace with your actual token
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/Gnanapravallika/Trusthworthy_OCTAI.git'

!git config user.email 'colab@trustoct.ai'
!git config user.name 'TrustOCT Colab'
!git pull $REPO_URL main --rebase

# Force-stage ONLY CSV predictions, metrics, figures, and YAMLs (skips 100MB .pth weights)
!git add -f outputs/resnet50/*.csv outputs/resnet50/*.png outputs/resnet50/*.yaml 2>/dev/null || true
!git add -f outputs/resnet50/layercam/ 2>/dev/null || true
!git add -f outputs/msf_resnet50/*.csv outputs/msf_resnet50/*.png outputs/msf_resnet50/*.yaml 2>/dev/null || true
!git add -f outputs/msf_resnet50/layercam/ 2>/dev/null || true

!git commit -m 'NB1: Save resnet50 + msf_resnet50 predictions and figures'
!git push $REPO_URL main
print('🚀 CSV prediction results and figures successfully pushed to GitHub!')
